<a href="https://colab.research.google.com/github/praveena0777/yolov8-visdrone-aerial-detection/blob/main/Copy_of_MiniProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install Ultralytics
!pip install ultralytics --quiet

from ultralytics import YOLO
import os

print("✅ YOLOv8 Installed Successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 83.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ YOLOv8 Installed Successfully!


In [ ]:
!pip install roboflow --quiet
from roboflow import Roboflow

print("✅ Roboflow Installed Successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.5 MB/s eta 0:00:00
✅ Roboflow Installed Successfully!


In [ ]:
# Paste your Private API Key between the quotes below
rf = Roboflow(api_key="dlG2Yu6F9vovyuVS2ALK")

print("✅ Roboflow connected successfully!")

✅ Roboflow connected successfully!


In [ ]:
# Download a good small object / aerial dataset
project = rf.workspace("visdrone-en8e0").project("visdrone-eqizo")
dataset = project.version(1).download("yolov8")

print("✅ Dataset Downloaded Successfully!")
print("Dataset location:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to VisDrone-1 in yolov8:: 100%|██████████| 14846/14846 [00:08<00:00, 1831.07it/s]


✅ Dataset Downloaded Successfully!
Dataset location: /content/VisDrone-1


In [ ]:
import os

train_images = len(os.listdir('/content/VisDrone-1/train/images'))
val_images = len(os.listdir('/content/VisDrone-1/valid/images'))
test_images = len(os.listdir('/content/VisDrone-1/test/images')) if os.path.exists('/content/VisDrone-1/test/images') else 0

print(f"Train images : {train_images}")
print(f"Valid images : {val_images}")
print(f"Test images  : {test_images}")
print(f"Total images : {train_images + val_images + test_images}")

Train images : 5180
Valid images : 1497
Test images  : 740
Total images : 7417


In [ ]:
import os
import shutil
import random

# Paths
src_train_img = '/content/VisDrone-1/train/images'
src_train_lbl = '/content/VisDrone-1/train/labels'
src_val_img   = '/content/VisDrone-1/valid/images'
src_val_lbl   = '/content/VisDrone-1/valid/labels'

# New subset folders
os.makedirs('/content/subset/train/images', exist_ok=True)
os.makedirs('/content/subset/train/labels', exist_ok=True)
os.makedirs('/content/subset/valid/images', exist_ok=True)
os.makedirs('/content/subset/valid/labels', exist_ok=True)

# Take 1500 train images
train_files = sorted(os.listdir(src_train_img))
random.seed(42)
selected_train = random.sample(train_files, 1500)

for f in selected_train:
    shutil.copy(os.path.join(src_train_img, f), '/content/subset/train/images/')
    lbl = f.rsplit('.', 1)[0] + '.txt'
    if os.path.exists(os.path.join(src_train_lbl, lbl)):
        shutil.copy(os.path.join(src_train_lbl, lbl), '/content/subset/train/labels/')

# Take 300 valid images
val_files = sorted(os.listdir(src_val_img))
selected_val = random.sample(val_files, 300)

for f in selected_val:
    shutil.copy(os.path.join(src_val_img, f), '/content/subset/valid/images/')
    lbl = f.rsplit('.', 1)[0] + '.txt'
    if os.path.exists(os.path.join(src_val_lbl, lbl)):
        shutil.copy(os.path.join(src_val_lbl, lbl), '/content/subset/valid/labels/')

print("✅ Subset created successfully!")
print(f"Train images: {len(os.listdir('/content/subset/train/images'))}")
print(f"Valid images: {len(os.listdir('/content/subset/valid/images'))}")

✅ Subset created successfully!
Train images: 1500
Valid images: 300


In [ ]:
import yaml

data = {
    'path': '/content/subset',
    'train': 'train/images',
    'val': 'valid/images',
    'nc': 10,   # VisDrone has 10 classes
    'names': ['pedestrian', 'people', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor']
}

with open('/content/subset/data.yaml', 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print("✅ data.yaml created successfully!")
print(open('/content/subset/data.yaml').read())

✅ data.yaml created successfully!
names:
- pedestrian
- people
- bicycle
- car
- van
- truck
- tricycle
- awning-tricycle
- bus
- motor
nc: 10
path: /content/subset
train: train/images
val: valid/images



In [ ]:
from ultralytics import YOLO

# Baseline Model - Normal YOLOv8s
model = YOLO("yolov8s.pt")

results = model.train(
    data='/content/subset/data.yaml',
    epochs=30,          # Reasonable for subset
    imgsz=640,
    batch=16,
    name='baseline_yolov8s',
    seed=42,
    patience=10
)

print("✅ Baseline Training Completed!")

Ultralytics 8.4.114 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/subset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_yolov8s, nbs=64, nms=False,

In [ ]:
from ultralytics import YOLO

# Train YOLOv8n (smaller model)
model = YOLO("yolov8n.pt")

results = model.train(
    data='/content/subset/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    name='baseline_yolov8n',
    seed=42,
    patience=10
)

print("✅ YOLOv8n Training Completed!")

Ultralytics 8.4.114 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/subset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_yolov8n, nbs=64, nms=False,

In [ ]:
import pandas as pd

yolov8n = pd.read_csv('/content/runs/detect/baseline_yolov8n/results.csv')
yolov8s = pd.read_csv('/content/runs/detect/baseline_yolov8s/results.csv')

print("="*60)
print("              FINAL RESULTS COMPARISON")
print("="*60)
print(f"YOLOv8n (Baseline)  → mAP50: {yolov8n['metrics/mAP50(B)'].max():.4f}  |  mAP50-95: {yolov8n['metrics/mAP50-95(B)'].max():.4f}")
print(f"YOLOv8s (Improved)  → mAP50: {yolov8s['metrics/mAP50(B)'].max():.4f}  |  mAP50-95: {yolov8s['metrics/mAP50-95(B)'].max():.4f}")
print("="*60)

improvement = yolov8s['metrics/mAP50(B)'].max() - yolov8n['metrics/mAP50(B)'].max()
print(f"Improvement in mAP50: +{improvement:.4f}")

              FINAL RESULTS COMPARISON
YOLOv8n (Baseline)  → mAP50: 0.3294  |  mAP50-95: 0.1180
YOLOv8s (Improved)  → mAP50: 0.4202  |  mAP50-95: 0.1611
Improvement in mAP50: +0.0907


In [ ]:
from ultralytics import YOLO

model = YOLO('/content/runs/detect/baseline_yolov8s/weights/best.pt')

results = model.predict(
    source='/content/subset/valid/images',
    save=True,
    conf=0.25,
    name='final_predictions'
)

print("✅ Inference Completed!")
print("Results saved in: /content/runs/detect/final_predictions")


image 1/300 /content/subset/valid/images/0000011_01350_d_0000003_jpg.rf.e86b92273027308ed7318d830581b2ab.jpg: 384x640 17 pedestrians, 6.4ms
image 2/300 /content/subset/valid/images/0000026_04500_d_0000033_jpg.rf.d279be8c47711fa6e2db3a7d034dd835.jpg: 384x640 (no detections), 6.4ms
image 3/300 /content/subset/valid/images/0000074_09738_d_0000019_jpg.rf.0536e1c05f601435281aaee7e0b3435f.jpg: 384x640 112 pedestrians, 6.4ms
image 4/300 /content/subset/valid/images/0000079_01062_d_0000003_jpg.rf.7d1163781984bbb87ea18370eef921c2.jpg: 384x640 14 pedestrians, 7.0ms
image 5/300 /content/subset/valid/images/0000079_02443_d_0000008_jpg.rf.6b5289f3ef811f681c7aa0d46cd724cb.jpg: 384x640 13 pedestrians, 6.4ms
image 6/300 /content/subset/valid/images/0000098_00016_d_0000001_jpg.rf.61ea0e9755d0e8275ca0a8dadf895dbf.jpg: 384x640 2 pedestrians, 6.4ms
image 7/300 /content/subset/valid/images/0000109_01140_d_0000063_jpg.rf.04b9420e3d02c85d173690fddc2c3c46.jpg: 384x640 4 pedestrians, 6.4ms
image 8/300 /conten

In [ ]:
import shutil

# Create zip of the important folders
shutil.make_archive('/content/baseline_yolov8n', 'zip', '/content/runs/detect/baseline_yolov8n')
shutil.make_archive('/content/baseline_yolov8s', 'zip', '/content/runs/detect/baseline_yolov8s')
shutil.make_archive('/content/final_predictions', 'zip', '/content/runs/detect/final_predictions')

print("✅ Zip files created! Now download them from the left panel.")

✅ Zip files created! Now download them from the left panel.
